# CSCU9M6 Assignment


In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

## Deliverable 1: Data Loading and Preprocessing

In [2]:
ROOT = Path.cwd()
TRAIN_CSV = ROOT / "sign_mnist_train.csv"
TEST_CSV = ROOT /  "sign_mnist_test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"dataset labels: {sorted(train_df['label'].unique())}")

dataset labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


### Fix the labels so that they are 0-23 rather than 0-24 and missing 9

In [3]:
def remap_labels(label):
    if label >= 10:
        return label - 1
    else:
        return label

train_df['label'] = train_df['label'].apply(remap_labels)
test_df['label'] = test_df['label'].apply(remap_labels)

TRAIN_CSV_UPDATED = ROOT / "sign_mnist_train_updated.csv"
TEST_CSV_UPDATED = ROOT /  "sign_mnist_test_updated.csv"

train_df.to_csv(TRAIN_CSV_UPDATED, index=False)
test_df.to_csv(TEST_CSV_UPDATED, index=False)

Create a Pytorch dataset which will use the updated CSV file

In [4]:
class SignLanguageDataset(Dataset):
    def __init__(self, dataframe):
        self.labels = dataframe['label'].astype(int).to_numpy()
        self.pixels = dataframe.drop(columns=['label']).values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.pixels[idx], dtype=torch.float32).view(1, 28, 28)
        x = x / 255.0 # normalize pixels to between 0 and 1
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [5]:
TEST_UPDATED_CSV = ROOT / "sign_mnist_test_updated.csv"
TRAIN_UPDATED_CSV = ROOT /  "sign_mnist_train_updated.csv"

train_df = pd.read_csv(TRAIN_UPDATED_CSV)
test_df = pd.read_csv(TEST_UPDATED_CSV)

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
    )

train_dataset = SignLanguageDataset(train_split_df)
val_dataset = SignLanguageDataset(val_split_df)
test_dataset = SignLanguageDataset(test_df)

print('Train size:', len(train_dataset))
print('Val size:', len(val_dataset))
print('Test size:', len(test_dataset))

print(f"dataset labels: {sorted(train_df['label'].unique())}")

Train size: 21964
Val size: 5491
Test size: 7172
dataset labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]


## Deliverable 2: CNN Architectures
- **Baseline CNN**: Simple CNN using 2 conv layers, 2 pooling layers, and 2 fully connected layers.
- **CNN with Residual Blocks**: Uses skip connections to allow for a deeper network without a vanishing gradient.
- **VGG Style Network**:

In [6]:
#Model Hyperparameters:

no_of_filters = [16,32]

blocks = 2

dropout = 0.5

In [7]:
class BaselineCNN(nn.Module):
    """ Basic CNN implementation """
    def __init__(self, num_classes=24):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Conv2d(1, no_of_filters[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(no_of_filters[1], no_of_filters[1] * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(no_of_filters[1] * 2 * 7 * 7, 128), # number of filters increased by multiple of 2, max pooling turned the image into 7x7 per filter (28 halved twice)
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.trunk(x)
        x = self.fc_layers(x)
        return x


class ResBlock(nn.Module):
    """ Residual block, uses 2 conv layers with batchnorm and relu, and during the forwarding a skip connection is used """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        res = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = x + res
        x = self.relu2(x)
        return x


class ResidualCNN(nn.Module):
    """ Deep CNN with residual blocks and increasing channel filters, before using average pooling and a fc layer """
    def __init__(self, num_classes=24, blocks=2):
        super().__init__()

        self.conv1 = nn.Conv2d(1, no_of_filters[1], kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(no_of_filters[1])
        self.relu1 = nn.ReLU()
        self.res_blocks1 = nn.ModuleList([ResBlock(no_of_filters[1]) for _ in range(blocks)])

        self.conv2 = nn.Conv2d(no_of_filters[1], no_of_filters[1] * 2, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(no_of_filters[1] * 2)
        self.relu2 = nn.ReLU()
        self.res_blocks2 = nn.ModuleList([ResBlock(no_of_filters[1] * 2) for _ in range(blocks)])

        self.conv3 = nn.Conv2d(no_of_filters[1] * 2, no_of_filters[1] * 4, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(no_of_filters[1] * 4)
        self.relu3 = nn.ReLU()
        self.res_blocks3 = nn.ModuleList([ResBlock(no_of_filters[1] * 4) for _ in range(blocks)])

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(no_of_filters[1] * 4, num_classes) # filter count has increased by multiple of 4, average pooling has made the image into a 1x1 single value per filter

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        for block in self.res_blocks1:
            x = block(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        for block in self.res_blocks2:
            x = block(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        for block in self.res_blocks3:
            x = block(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class VGGNetwork(nn.Module):
    """ VGG style CNN with stages using 2 conv layers with max pooling and dropout before two fc layers """
    def __init__(self, num_classes=24):
        super().__init__()

        self.trunk = nn.Sequential(
            nn.Conv2d(1, no_of_filters[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(no_of_filters[1], no_of_filters[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(no_of_filters[1], no_of_filters[1] * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(no_of_filters[1] * 2, no_of_filters[1] * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(no_of_filters[1] * 2, no_of_filters[1] * 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(no_of_filters[1] * 4, no_of_filters[1] * 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(no_of_filters[1] * 4 * 3 * 3, 512), # filter count has increased by multiple of 4, max pooling has made image into 3x3 per filter
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.trunk(x)
        x = self.fc_layers(x)
        return x

## Deliverable 3: Training, Testing, and Evaluation

### Hyperparameters

In [13]:
lrs = [0.001,0.005,0.0001]
batch_sizes = [16,32,64]
no_of_epochs = [10,15,20]
lr = lrs[0]
batch_size = batch_sizes[2]
epochs = no_of_epochs[0]

### Training, validation and testing

In [14]:
baseline_model = BaselineCNN()
resnet_model = ResidualCNN()
vgg_model = VGGNetwork()

criterion = nn.CrossEntropyLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

baseline_model.to(device)
resnet_model.to(device)
vgg_model.to(device)

baseline_optimizer = optim.Adam(baseline_model.parameters(), lr)
resnet_optimizer = optim.Adam(resnet_model.parameters(), lr)
vgg_optimizer = optim.Adam(vgg_model.parameters(), lr)

train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size, shuffle=False)

def validate_model(model, criterion, data_loader):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0) # calculate loss
            running_correct += (preds == labels).sum().item()# accuracy
            total += labels.size(0)# total datapoints seen

    avg_loss = running_loss / total
    accuracy = running_correct / total
    return avg_loss, accuracy

def train_model(model, optimizer, criterion, train_loader, val_loader, epochs):
    training_metrics = {
        'loss': [],
        'accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        for inputs, labels in train_loader:
            optimizer.zero_grad()
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

        # store metrics in dictionary to be used for evaluation
        train_loss = running_loss / total
        train_acc = running_correct / total
        training_metrics['loss'].append(train_loss)
        training_metrics['accuracy'].append(train_acc)

        val_loss, val_acc = validate_model(model, criterion, val_loader)
        training_metrics['val_loss'].append(val_loss)
        training_metrics['val_accuracy'].append(val_acc)

        print(
            f"Epoch {epoch+1}/{epochs} "
            f"train loss: {train_loss:.4f} - train accuracy: {train_acc:.4f} "
            f"val loss: {val_loss:.4f} - val accuracy: {val_acc:.4f}"
        )

    return training_metrics

print("Baseline CNN: ")
baseline_metrics = train_model(baseline_model, baseline_optimizer, criterion, train_loader, val_loader, epochs)

print("ResNet: ")
resnet_metrics = train_model(resnet_model, resnet_optimizer, criterion, train_loader, val_loader, epochs)

print("VGG: ")
vgg_metrics = train_model(vgg_model, vgg_optimizer, criterion, train_loader, val_loader, epochs)

#torch.save(baseline_model.state_dict(), ROOT / 'baseline_model.pth')
#torch.save(resnet_model.state_dict(), ROOT / 'residual_model.pth')
#torch.save(vgg_model.state_dict(), ROOT / 'vgg_model.pth')

cuda
Baseline CNN: 
Epoch 1/10 train loss: 1.2975 - train accuracy: 0.6080 val loss: 0.4022 - val accuracy: 0.8887
Epoch 2/10 train loss: 0.2287 - train accuracy: 0.9360 val loss: 0.1082 - val accuracy: 0.9718
Epoch 3/10 train loss: 0.0593 - train accuracy: 0.9881 val loss: 0.0261 - val accuracy: 0.9980
Epoch 4/10 train loss: 0.0174 - train accuracy: 0.9981 val loss: 0.0073 - val accuracy: 1.0000
Epoch 5/10 train loss: 0.0068 - train accuracy: 0.9996 val loss: 0.0049 - val accuracy: 1.0000
Epoch 6/10 train loss: 0.0028 - train accuracy: 1.0000 val loss: 0.0025 - val accuracy: 1.0000
Epoch 7/10 train loss: 0.0017 - train accuracy: 1.0000 val loss: 0.0018 - val accuracy: 1.0000
Epoch 8/10 train loss: 0.0012 - train accuracy: 1.0000 val loss: 0.0015 - val accuracy: 1.0000
Epoch 9/10 train loss: 0.0008 - train accuracy: 1.0000 val loss: 0.0007 - val accuracy: 1.0000
Epoch 10/10 train loss: 0.0374 - train accuracy: 0.9880 val loss: 0.0622 - val accuracy: 0.9783
ResNet: 
Epoch 1/10 train los

In [10]:
def test_model(model, criterion, data_loader):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0
    all_preds = [] # save predictions and labels for confusion matrix
    all_labels = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    avg_loss = running_loss / len(data_loader)
    accuracy = running_correct / total
    return avg_loss, accuracy, all_labels, all_preds

baseline_test_loss, baseline_test_acc, baseline_labels, baseline_preds = test_model(baseline_model, criterion, test_loader)
resnet_test_loss, resnet_test_acc, resnet_labels, resnet_preds = test_model(resnet_model, criterion, test_loader)
vgg_test_loss, vgg_test_acc, vgg_labels, vgg_preds = test_model(vgg_model, criterion, test_loader)

print(f"Baseline test loss: {baseline_test_loss:.4f}")
print(f"Baseline test accuracy: {baseline_test_acc:.4f}")
print(f"ResNet test loss: {resnet_test_loss:.4f}")
print(f"ResNet test accuracy: {resnet_test_acc:.4f}")
print(f"VGG test loss: {vgg_test_loss:.4f}")
print(f"VGG test accuracy: {vgg_test_acc:.4f}")

Baseline test loss: 28.2294
Baseline test accuracy: 0.9052
ResNet test loss: 0.2643
ResNet test accuracy: 0.9996
VGG test loss: 12.8022
VGG test accuracy: 0.9406
